# Molecular simulation in Colab

Select a **CPU runtime** and use Runtime → Run all for H₂/STO-3G and integral export. QE installation and browser downloads are opt-in below. To try GPU4PySCF, select an NVIDIA GPU runtime and follow the optional GPU cell. TPU is not supported. Runtime files are temporary; download results or copy them to Drive.

## Get the project
Use an existing checkout, upload the project to `/content/colabexercise`, or clone GitHub. `COLABEXERCISE_REF` can select a published branch; the default is `main`. Thread limits are set before importing numerical libraries.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import tempfile

for key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[key] = '1'
runtime_root = Path('/content') if Path('/content').is_dir() else Path(tempfile.gettempdir())
project = Path.cwd() if (Path.cwd() / 'quantum_workflow/workflow.py').is_file() else runtime_root / 'colabexercise'
repo_url = 'https://github.com/gutama/colabexercise.git'
branch = os.environ.get('COLABEXERCISE_REF', 'main')
if not project.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', branch, repo_url, str(project)], check=True)
if not (project / 'pyproject.toml').is_file():
    raise RuntimeError('Project files are missing; upload the project or select its published branch.')
artifact_root = Path(os.environ.get('COLABEXERCISE_OUTPUT', str(runtime_root)))
artifact_root.mkdir(parents=True, exist_ok=True)
print('Python:', sys.version.split()[0], 'Project:', project)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(project)], check=True)

## Optional NVIDIA GPU setup
Uncomment the installation below only if the runtime has a compatible CUDA 12 NVIDIA GPU. Consult the [GPU4PySCF installation guide](https://pyscf.org/user/gpu.html) for the current package versions and other CUDA versions. Restart the runtime if Colab requests it. GPU use can fail if the installed CUDA stack or calculation is unsupported; it is not silently replaced by CPU. This path has not been tested on a GPU in this project.

In [ ]:
# !nvidia-smi
# %pip install gpu4pyscf-cuda12x cutensor-cu12

## PySCF: H2 and integral export
Change the example JSON or the in-memory config for other systems. Set `device` to `gpu` after installing the GPU stack, or leave it at `cpu`.

In [ ]:
from quantum_workflow.workflow import load_config, run

config = load_config(project / 'examples/h2-pyscf.json')
config.device = 'cpu'  # 'gpu' requires GPU4PySCF and a compatible GPU
pyscf_output = Path(tempfile.mkdtemp(prefix='pyscf-', dir=artifact_root))
summary = run(config, pyscf_output)
print(summary)
print('Output:', pyscf_output)

## Optional QE calculation (CPU)
Downloads micromamba and installs a separate conda-forge QE 7.6 CPU environment, plus an example H PBE pseudopotential. This takes several minutes and disk space. QE uses a periodic supercell; independently test vacuum and cutoffs. Review downloaded software and pseudopotential suitability and provenance before scientific use. This section does not export VQE-ready integrals.

In [ ]:
RUN_QE = os.environ.get('COLABEXERCISE_RUN_QE', '0') == '1'  # Set True to opt in.
if RUN_QE:
    import tarfile
    import shutil

    mamba = runtime_root / 'micromamba'
    qe_env = runtime_root / 'qe-env'
    if not (qe_env / 'bin/pw.x').exists():
        subprocess.run(['curl', '-fL', 'https://micro.mamba.pm/api/micromamba/linux-64/latest', '-o', str(runtime_root / 'micromamba.tar.bz2')], check=True)
        with tarfile.open(str(runtime_root / 'micromamba.tar.bz2'), 'r:bz2') as tar:
            source = tar.extractfile('bin/micromamba')
            if source is None:
                raise RuntimeError('micromamba archive is missing its executable')
            with source, mamba.open('wb') as target:
                shutil.copyfileobj(source, target)
        mamba.chmod(0o755)
        subprocess.run([str(mamba), 'create', '-y', '-r', str(runtime_root / 'mamba-root'), '-p', str(qe_env), '-c', 'conda-forge', '--override-channels', 'qe=7.6'], check=True)
    os.environ['PATH'] = str(qe_env / 'bin') + os.pathsep + os.environ['PATH']
    pseudo_dir = project / 'pseudopotentials'
    pseudo_dir.mkdir(exist_ok=True)
    pseudo = pseudo_dir / 'H.pbe-kjpaw_psl.1.0.0.UPF'
    subprocess.run(['curl', '-fL', 'https://pseudopotentials.quantum-espresso.org/upf_files/H.pbe-kjpaw_psl.1.0.0.UPF', '-o', str(pseudo)], check=True)
    qe_config = load_config(project / 'examples/h2-qe.json')
    qe_output = Path(tempfile.mkdtemp(prefix='qe-', dir=artifact_root))
    print(run(qe_config, qe_output))
    print('Output:', qe_output)
else:
    print('QE skipped. Set RUN_QE = True to install QE and run the CPU example.')

## Download PySCF results (opt-in)
The archive includes integrals and metadata. Set `DOWNLOAD_RESULTS=True` to open a Colab browser download. For QE, archive `qe_output` similarly; its scratch directory can be large. See the README for tensor conventions and an FCI consumption example.

In [ ]:
DOWNLOAD_RESULTS = False  # True for a browser download in Google Colab.
if DOWNLOAD_RESULTS:
    import shutil
    from google.colab import files

    archive = shutil.make_archive(str(artifact_root / 'pyscf-results'), 'zip', pyscf_output)
    files.download(archive)
else:
    print('Results saved:', pyscf_output)